# Facial Visual Profile — RQ4 Controlled Training Study

**One Notebook · 6 experimental conditions × 3 seeds = 18 independent trainings · resume-safe early stopping**

**Research question:** *Why does accurate explicit geometry fail to provide consistent gains in facial attribute recognition under image corruption?*

Built against the project artifacts actually found in **Google Drive → FacialVisualProfile** and the archived CelebA / model preprocessing code. **New experimental code; not claimed to have run on the full Colab GPU/CelebA data.** Review the pre-flight checks and run a smoke test first.

| ID | Input | Training distribution | Whole-RGB feature masking |
|:--|:--|:--|:--|
| A | R18 RGB-only | Clean-only | No |
| B | R18 + GT five-point geometry | Clean-only | No |
| C | R18 RGB-only | 50% clean / 50% corrupted | No |
| D | R18 + GT five-point geometry | 50% clean / 50% corrupted | No |
| E | R18 RGB-only | Clean-only | 20% of samples |
| F | R18 + GT five-point geometry | Clean-only | 20% of samples |

**No Constant Geometry, R50, or MTCNN.** GT under occlusion remains privileged input. These are new, matched-protocol runs; old benchmark scores are background context only.

**Safe restart:** outputs go to a *new* Google Drive folder; each run keeps `latest.pt`, `best_model.pt`, progress history and finished markers; re-running the notebook resumes completed epochs/runs. If using temporary Colab disk, re-mount Drive and run cells in order after disconnect.

**Pre-registered analysis:** run all configurations, select checkpoint by clean-validation mAP, fit thresholds on clean validation only, then run fixed clean + 20-corruption test. Do not tune on test results.

**Revision 2026-10-22 — Resume-safe early stopping:** Keep `RUN_STUDY`, `EPOCHS=10`, the original `study_protocol.json`, and all existing run configurations unchanged. New stopping logic uses minimum 4 epochs, patience 2, monitored on clean-validation mAP, with a separate and persistent `early_stopping_policy.json`. Existing checkpoints such as `A_seed0/latest.pt` resume without restarting. Do not run the old and new notebook simultaneously.


In [ ]:
# 1 — Editable settings. DO NOT silently change these after starting real runs.
PROJECT_PATH = '/content/drive/MyDrive/FacialVisualProfile'
RUN_STUDY = 'rq4_training_v1'    # create an independent new folder; never overwrite old runs
SEEDS = (0, 1, 2)
EPOCHS = 10           # UNCHANGED: maximum epochs; must match original study_protocol.json
EARLY_STOPPING = True
MIN_EPOCHS = 4
EARLY_STOP_PATIENCE = 2  # Stop after 2 consecutive epochs without improvement, after min epochs
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
NUM_WORKERS = 2
RGB_MASK_PROB = 0.20       # Entire 512-D RGB feature vector zeroed for 20% of training samples in E/F
CORRUPTED_TRAIN_PROB = 0.50
RUN_FINAL_TEST = True      # Enabled only after all 18 trainings + threshold exports are complete
EXTRACT_TEST_IMAGES = True
SMOKE_ONLY = False         # First use True to run short data/model checks, then False for full experiment

EXPERIMENTS = {
    'A': dict(geometry=False, augmentation=False, mask_rgb=False),
    'B': dict(geometry=True,  augmentation=False, mask_rgb=False),
    'C': dict(geometry=False, augmentation=True,  mask_rgb=False),
    'D': dict(geometry=True,  augmentation=True,  mask_rgb=False),
    'E': dict(geometry=False, augmentation=False, mask_rgb=True),
    'F': dict(geometry=True,  augmentation=False, mask_rgb=True),
}
assert len(EXPERIMENTS) * len(SEEDS) == 18

In [ ]:
# 2 — Environment / Google Drive mount and persistent paths
import os, sys, json, hashlib, math, random, time, shutil, zipfile, io, platform
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TF
from PIL import Image, ImageFilter, ImageEnhance
from sklearn.metrics import average_precision_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running inside Colab; Google Drive must already be mounted.')

PROJECT = Path(PROJECT_PATH)
assert PROJECT.is_dir(), f'Missing mounted Google Drive project: {PROJECT}'
assert torch.cuda.is_available() or SMOKE_ONLY, 'Select a GPU runtime before full training'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STUDY = PROJECT / 'runs' / RUN_STUDY
STUDY.mkdir(parents=True, exist_ok=True)
RAW = PROJECT / 'data'
LOCAL = Path('/content/fvp_rq4_data')
LOCAL.mkdir(parents=True, exist_ok=True)
IMAGE_ROOT = LOCAL / 'img_align_celeba'
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

ATTRIBUTES = [
    'Smiling','Mouth_Slightly_Open','Eyeglasses','Bangs','Wearing_Hat','Mustache',
    'No_Beard','Black_Hair','Bushy_Eyebrows','Arched_Eyebrows','High_Cheekbones',
    'Bags_Under_Eyes','Big_Lips','Big_Nose','Double_Chin','Oval_Face',
    'Receding_Hairline','Blond_Hair','Brown_Hair','Gray_Hair','Bald','Straight_Hair',
    'Wavy_Hair','Sideburns'
]
assert len(ATTRIBUTES) == 24

PROTOCOL = dict(study=RUN_STUDY,seeds=SEEDS,experiments=EXPERIMENTS,
                epochs=EPOCHS,batch_size=BATCH_SIZE,lr=LEARNING_RATE,
                weight_decay=WEIGHT_DECAY,num_workers=NUM_WORKERS,
                rgb_mask_prob=RGB_MASK_PROB,aug_prob=CORRUPTED_TRAIN_PROB,
                attributes=ATTRIBUTES, image_size=224,
                corruption_policy='five families x four levels, 50-percent-clean stochastic augmentation',
                pretrained='ResNet18_Weights.IMAGENET1K_V1',
                checkpoint_selection='maximum clean-validation mAP; earliest epoch on exact tie')
def write_json(path, data):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.partial')
    temp.write_text(json.dumps(data,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
    temp.replace(path)

def write_torch(path, data):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.partial')
    torch.save(data,temp); temp.replace(path)

proto_path=STUDY/'study_protocol.json'
if proto_path.exists():
    assert json.loads(proto_path.read_text()) == json.loads(json.dumps(PROTOCOL)), (
        'Protocol has changed; choose a new RUN_STUDY name instead of mixing runs')
else:
    write_json(proto_path, PROTOCOL)
# NEW: Persist early-stopping settings separately. Do NOT add them to PROTOCOL,
# otherwise an already-started rq4_training_v1 would reject its own study_protocol.json.
EARLY_STOP_POLICY = {
    'enabled': bool(EARLY_STOPPING),
    'max_epochs': int(EPOCHS),
    'min_epochs': int(MIN_EPOCHS),
    'patience': int(EARLY_STOP_PATIENCE),
    'monitor': 'clean_validation_mAP',
    'improvement_rule': 'strict > previous best (ties do not count)',
    'on_existing_history': 'read original best.json/latest.pt and use completed epochs',
    'original_protocol_unchanged': True,
}
assert EPOCHS == 10, 'Keep EPOCHS=10 for compatibility with existing study_protocol.json'
assert 1 <= MIN_EPOCHS <= EPOCHS
assert EARLY_STOP_PATIENCE >= 1
policy_file=STUDY/'early_stopping_policy.json'
if policy_file.exists():
    assert json.loads(policy_file.read_text()) == EARLY_STOP_POLICY, (
        'Early stopping policy changed inside existing study. Restore original policy.')
else:
    write_json(policy_file,EARLY_STOP_POLICY)
print('Early stopping policy:', EARLY_STOP_POLICY)

write_json(STUDY/'runtime_latest.json', {'python':platform.python_version(),
           'torch':torch.__version__,'torchvision':__import__('torchvision').__version__,
           'Pillow':__import__('PIL').__version__,'device':str(DEVICE),
           'cuda':torch.version.cuda})
print('Mounted project:',PROJECT,'| GPU:',DEVICE,'| New study:',STUDY)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Early stopping policy: {'enabled': True, 'max_epochs': 10, 'min_epochs': 4, 'patience': 2, 'monitor': 'clean_validation_mAP', 'improvement_rule': 'strict > previous best (ties do not count)', 'on_existing_history': 'read original best.json/latest.pt and use completed epochs', 'original_protocol_unchanged': True}
Mounted project: /content/drive/MyDrive/FacialVisualProfile | GPU: cuda | New study: /content/drive/MyDrive/FacialVisualProfile/runs/rq4_training_v1


In [ ]:
# 3 — Prepare CelebA from the verified project Drive layout
# Only reads the original data archive and three text annotations; no historical checkpoints are modified.
required = ['img_align_celeba.zip','list_eval_partition.txt','list_attr_celeba.txt','list_landmarks_align_celeba.txt']
for filename in required:
    assert (RAW/filename).is_file(), f'Missing original project file: {RAW/filename}'

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):
            h.update(block)
    return h.hexdigest()

# Original old notebook's expected image ZIP checksum. Fail if the archive is not the same.
EXPECTED_ZIP_SHA = '46fb89443c578308acf364d7d379fe1b9efb793042c0af734b6112e4fd3a8c74'
assert sha256_file(RAW/'img_align_celeba.zip') == EXPECTED_ZIP_SHA, 'CelebA ZIP differs from original project'

zip_path=LOCAL/'img_align_celeba.zip'
if not zip_path.exists() or zip_path.stat().st_size != (RAW/'img_align_celeba.zip').stat().st_size:
    print('Copying original CelebA ZIP to Colab local disk (about 1.44 GB)...')
    tmp=zip_path.with_suffix('.zip.partial'); shutil.copyfile(RAW/'img_align_celeba.zip',tmp); tmp.replace(zip_path)
assert sha256_file(zip_path) == EXPECTED_ZIP_SHA

partition_rows=[line.split() for line in (RAW/'list_eval_partition.txt').read_text(encoding='utf-8-sig').splitlines()]
assert all(len(row)==2 for row in partition_rows)
partitions=dict(partition_rows)
assert len(partitions)==202599 and Counter(partitions.values()) == {'0':162770,'1':19867,'2':19962}

# Extract once. ZIP member paths are flattened to exactly the official filenames.
expected=set(partitions)
existing=sum(1 for name in expected if (IMAGE_ROOT/name).is_file())
if existing<len(expected):
    with zipfile.ZipFile(zip_path) as z:
        members={}
        for v in z.infolist():
            if v.is_dir(): continue
            name=Path(v.filename).name
            assert name.lower().endswith('.jpg') and name not in members
            members[name]=v
        assert set(members)==expected,'Image archive does not match official partition names'
        targets=expected if EXTRACT_TEST_IMAGES and not SMOKE_ONLY else {n for n,k in partition_rows if k!='2'}
        for i,name in enumerate(sorted(targets),1):
            dst=IMAGE_ROOT/name
            if not dst.exists():
                with z.open(members[name]) as src, open(dst.with_suffix('.jpg.partial'),'wb') as out:
                    shutil.copyfileobj(src,out)
                dst.with_suffix('.jpg.partial').replace(dst)
            if i%40000==0: print('Images ready:',i,'/',len(targets))
print('CelebA local extraction:',sum(1 for n in expected if (IMAGE_ROOT/n).exists()),'images')

CelebA local extraction: 202599 images


In [ ]:
# 4 — Parse labels and five-point GT, respecting original 24-label order
with open(RAW/'list_attr_celeba.txt',encoding='utf-8-sig') as f:
    count=int(next(f).strip()); columns=next(f).split()
    selected=[columns.index(a) for a in ATTRIBUTES]
    y_map={}
    for line in f:
        v=line.split(); y_map[v[0]]=np.asarray([float(int(v[1+i])==1) for i in selected],dtype=np.float32)
with open(RAW/'list_landmarks_align_celeba.txt',encoding='utf-8-sig') as f:
    count_l=int(next(f).strip()); next(f)
    xy_map={}
    for line in f:
        v=line.split();xy_map[v[0]]=np.asarray([float(x) for x in v[1:11]],dtype=np.float32)
assert count==count_l==len(y_map)==len(xy_map)==202599
assert set(y_map)==set(xy_map)==set(partitions)
SPLITS={s:[name for name,k in partition_rows if k==v] for s,v in [('train','0'),('valid','1'),('test','2')]}
assert [len(SPLITS[s]) for s in ['train','valid','test']] == [162770,19867,19962]
print('Verified CelebA official splits:',{k:len(v) for k,v in SPLITS.items()},'| attributes:',len(ATTRIBUTES))

Verified CelebA official splits: {'train': 162770, 'valid': 19867, 'test': 19962} | attributes: 24


In [ ]:
# 5 — Reproduce original 224x224 letterbox + exact pixel-center landmark mapping
SIZE=224
MEAN=(0.485,0.456,0.406)
STD=(0.229,0.224,0.225)
FAMILIES=('blur','brightness','rotation','occlusion','jpeg')
LEVELS={
    'blur':(1,2,3,4), 'brightness':(.8,.6,.4,.2),
    'rotation':(5,10,15,20), 'occlusion':(.10,.20,.30,.40),
    'jpeg':(75,50,25,10)
}

def prepare_canvas(image, raw_points):
    width,height=image.size
    scale=SIZE/max(width,height)
    new_width,new_height=round(width*scale),round(height*scale)
    left,top=(SIZE-new_width)//2,(SIZE-new_height)//2
    resized=image.convert('RGB').resize((new_width,new_height),Image.Resampling.BILINEAR)
    canvas=Image.new('RGB',(SIZE,SIZE),(0,0,0))
    canvas.paste(resized,(left,top))
    points=np.array(raw_points,dtype=np.float32).reshape(5,2).copy()
    points[:,0]=(points[:,0]+0.5)*new_width/width-0.5+left
    points[:,1]=(points[:,1]+0.5)*new_height/height-0.5+top
    return canvas,points

def direction_for(filename):
    payload=f'day14_draft_v1|0|{filename}|rotation'.encode()
    return +1 if (hashlib.sha256(payload).digest()[0] & 1) else -1

def transform_corruption(canvas, xy, family=None, severity=0, filename=''):
    if family is None or severity==0: return canvas, xy.copy()
    assert family in FAMILIES and severity in (1,2,3,4)
    value=LEVELS[family][severity-1]
    result=xy.copy()
    if family=='blur':
        canvas=canvas.filter(ImageFilter.GaussianBlur(radius=value))
    elif family=='brightness':
        canvas=ImageEnhance.Brightness(canvas).enhance(value)
    elif family=='rotation':
        degrees=direction_for(filename)*value
        canvas=canvas.rotate(degrees,resample=Image.Resampling.BILINEAR,expand=False,center=(112.0,112.0),fillcolor=(0,0,0))
        radians=np.deg2rad(degrees)
        c,s=np.cos(radians),np.sin(radians)
        offset=result-np.array([111.5,111.5],dtype=np.float32)
        result=np.stack([111.5+c*offset[:,0]+s*offset[:,1],
                         111.5-s*offset[:,0]+c*offset[:,1]],axis=1).astype(np.float32)
    elif family=='occlusion':
        side=int(math.floor(SIZE*value+0.5))
        x0=112-side//2;y0=168-side//2
        canvas=canvas.copy()
        canvas.paste((0,0,0),(x0,y0,x0+side,y0+side))
    elif family=='jpeg':
        stream=io.BytesIO()
        canvas.save(stream,format='JPEG',quality=value,subsampling=2,optimize=False,progressive=False)
        stream.seek(0)
        with Image.open(stream) as pic:
            canvas=pic.convert('RGB')
    return canvas, result

def normalize_geometry(points):
    return torch.as_tensor((2*np.asarray(points,dtype=np.float32)/(SIZE-1)-1).reshape(-1).copy(),dtype=torch.float32)

def augment_choice(filename,seed,epoch):
    # Per-sample, per-epoch deterministic decision independent of experiment ID & DataLoader worker count.
    raw=hashlib.sha256(f'RQ4v1|{seed}|{epoch}|{filename}'.encode()).digest()
    q=int.from_bytes(raw[:8],'big')/(2**64)
    if q>=CORRUPTED_TRAIN_PROB:return None,0
    family=FAMILIES[raw[8]%len(FAMILIES)]
    severity=1+(raw[9]%4)
    return family,severity

class FacialDataset(Dataset):
    def __init__(self,split,augment=False,seed=0,epoch=0,family=None,severity=0):
        self.names=SPLITS[split];self.augment=augment;self.seed=seed;self.epoch=epoch
        self.family=family;self.severity=severity
    def __len__(self):return len(self.names)
    def __getitem__(self,index):
        name=self.names[index]
        with Image.open(IMAGE_ROOT/name) as img:
            canvas,pts=prepare_canvas(img,xy_map[name])
        if self.augment:
            family,severity=augment_choice(name,self.seed,self.epoch)
        else:
            family,severity=self.family,self.severity
        canvas,pts=transform_corruption(canvas,pts,family,severity,name)
        x=TF.normalize(TF.to_tensor(canvas),MEAN,STD)
        return x,torch.from_numpy(y_map[name].copy()),normalize_geometry(pts)

In [ ]:
# 6 — Model factory: original R18 and Oracle architecture, with optional whole-branch masking
class WholeRGBMask(nn.Module):
    def __init__(self,p=0.):
        super().__init__();self.p=float(p)
    def forward(self,x):
        if self.training and self.p>0:
            mask=torch.rand((len(x),1),device=x.device)<self.p
            return x.masked_fill(mask,0.0)   # NO inverted-dropout scaling
        return x

class R18Attributes(nn.Module):
    def __init__(self,pretrained=True,mask_rgb=False):
        super().__init__()
        self.network=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.network.fc=nn.Identity()
        self.rgb_mask=WholeRGBMask(RGB_MASK_PROB if mask_rgb else 0.)
        self.classifier=nn.Linear(512,24)
    def forward(self,x,points=None):
        features=self.rgb_mask(self.network(x))
        return self.classifier(features)

class OracleAttributes(nn.Module):
    def __init__(self,pretrained=True,mask_rgb=False):
        super().__init__()
        self.image_encoder=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_encoder.fc=nn.Identity()
        self.rgb_mask=WholeRGBMask(RGB_MASK_PROB if mask_rgb else 0.)
        self.geometry_encoder=nn.Sequential(nn.Linear(10,32),nn.ReLU(),nn.Linear(32,32),nn.ReLU())
        self.classifier=nn.Linear(544,24)
    def forward(self,x,points):
        rgb=self.rgb_mask(self.image_encoder(x))
        geo=self.geometry_encoder(points)
        return self.classifier(torch.cat((rgb,geo),dim=1))

def make_model(exp,pretrained=True):
    opts=EXPERIMENTS[exp]
    cls=OracleAttributes if opts['geometry'] else R18Attributes
    return cls(pretrained=pretrained,mask_rgb=opts['mask_rgb'])

def run_forward(model, batch):
    images,labels,pts=batch
    return model(images,pts)

# Smoke-test all six model modes, tensor shapes & test-time dropout disabling.
if SMOKE_ONLY:
    t=torch.randn(2,3,224,224,device=DEVICE)
    p=torch.randn(2,10,device=DEVICE)
    for name in EXPERIMENTS:
        m=make_model(name,pretrained=False).to(DEVICE)
        m.eval()
        with torch.no_grad():out=m(t,p)
        assert out.shape==(2,24) and torch.isfinite(out).all()
        del m
    print('MODEL SMOKE: PASS')

In [ ]:
# 7 — Metrics, deterministic loaders and clean-validation threshold fitting
from sklearn.metrics import f1_score

def set_seed(s):
    random.seed(s);np.random.seed(s);torch.manual_seed(s)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def seed_worker(worker_id):
    w=torch.initial_seed()%(2**32)
    np.random.seed(w);random.seed(w)

def make_loader(ds,shuffle=False,seed=0,eval_batch=128):
    return DataLoader(ds,batch_size=BATCH_SIZE if shuffle else eval_batch,shuffle=shuffle,
                      num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),
                      worker_init_fn=seed_worker,
                      generator=torch.Generator().manual_seed(seed),persistent_workers=False)

def compute_metrics(y,prob,thresholds=None):
    if thresholds is None:thresholds=np.full(24,.5,dtype=np.float64)
    aps=[];fs=[]
    for a in range(24):
        assert np.unique(y[:,a]).size==2, f'Class missing for attribute {ATTRIBUTES[a]}'
        aps.append(float(average_precision_score(y[:,a],prob[:,a])))
        fs.append(float(f1_score(y[:,a],prob[:,a]>=thresholds[a],zero_division=0)))
    return {'mAP':float(np.mean(aps)),'macro_f1':float(np.mean(fs)),
            'AP_by_attr':aps,'F1_by_attr':fs,'n':len(y)}

def fit_thresholds(y,prob):
    grid=np.arange(5,96,dtype=np.float64)/100
    threshold=[]
    for a in range(24):
        gold=y[:,a].astype(bool)
        preds=prob[:,a,None]>=grid[None,:]
        tp=(preds & gold[:,None]).sum(axis=0)
        fp=(preds & ~gold[:,None]).sum(axis=0)
        fn=((~preds) & gold[:,None]).sum(axis=0)
        den=2*tp+fp+fn
        score=np.divide(2*tp,den,out=np.zeros_like(den,dtype=float),where=den>0)
        best=np.flatnonzero(np.isclose(score,score.max(),rtol=0,atol=1e-12))
        ix=sorted(best,key=lambda i:(abs(grid[i]-.5),grid[i]))[0]
        threshold.append(float(grid[ix]))
    return np.array(threshold,dtype=np.float64)

@torch.inference_mode()
def predict(model,ds,desc='inference'):
    model.eval()
    prob_parts=[];true_parts=[]
    for i,batch in enumerate(make_loader(ds,shuffle=False,seed=0)):
        x,y,p=batch
        z=run_forward(model,(x.to(DEVICE,non_blocking=True),y,p.to(DEVICE,non_blocking=True)))
        assert z.shape==(len(x),24) and torch.isfinite(z).all()
        prob_parts.append(torch.sigmoid(z).cpu().numpy())
        true_parts.append(y.numpy())
        if i>0 and i%80==0: print(f'{desc}: {sum(len(q) for q in true_parts)}/{len(ds)}',flush=True)
    return np.concatenate(true_parts),np.concatenate(prob_parts)

In [ ]:
# 8 — Training with resume/checkpoint-selection; no test labels are accessed in this section
LOSS=nn.BCEWithLogitsLoss()

def early_stop_status(history,best_epoch):
    """Return (stop_now, stale_epochs). Tracks actual best epoch, including history from old notebook."""
    if not history:
        return False,0
    epoch=int(history[-1]['epoch'])
    assert 1 <= best_epoch <= epoch
    stale_epochs=epoch-best_epoch  # ties are counted as non-improvement
    stop_now=(EARLY_STOPPING and epoch>=MIN_EPOCHS and stale_epochs>=EARLY_STOP_PATIENCE)
    return bool(stop_now), int(stale_epochs)

def mark_training_complete(folder,history,best,reason):
    """Only mark complete AFTER latest.pt and best_model.pt exist and agree with history."""
    assert history and (folder/'latest.pt').exists() and (folder/'best_model.pt').exists()
    final_epoch=int(history[-1]['epoch'])
    assert final_epoch<=EPOCHS
    assert reason in ('early_stopping','max_epochs')
    assert (reason!='early_stopping') or early_stop_status(history,int(best['epoch']))[0]
    write_json(folder/'train_complete.json', {
        'epochs':final_epoch,      # actual executed epochs, not the previous required 10
        'max_epochs':EPOCHS,
        'stop_reason':reason,
        'best':best,
        'early_stopping':EARLY_STOP_POLICY,
    })
    print('TRAINING COMPLETE:',folder.name,'ran',final_epoch,'epochs;',
          'best epoch',best['epoch'],'reason:',reason,flush=True)

def train_one(exp,seed):
    key=f'{exp}_seed{seed}'
    folder=STUDY/key
    folder.mkdir(parents=True,exist_ok=True)
    expected={'exp':exp,'seed':seed,'options':EXPERIMENTS[exp], 'protocol':PROTOCOL}
    config_file=folder/'config.json'
    if config_file.exists():
        assert json.loads(config_file.read_text()) == json.loads(json.dumps(expected)),'Run config differs, stop'
    else:write_json(config_file,expected)

    # Seed and instantiate as before; resumed weights / optimizer overwrite initial values.
    set_seed(seed)
    model=make_model(exp,pretrained=True).to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    latest=folder/'latest.pt'
    history=[];start=1
    if latest.exists():
        obj=torch.load(latest,map_location='cpu',weights_only=True)
        assert obj['exp']==exp and obj['seed']==seed
        model.load_state_dict(obj['model'],strict=True)
        optimizer.load_state_dict(obj['optimizer'])
        history=obj['history'];start=int(obj['epoch'])+1
        assert [x['epoch'] for x in history] == list(range(1,start))
        del obj
        print('RESUME:',key,'completed epochs',start-1,'next',start,flush=True)
    elif (folder/'history.json').exists() or (folder/'best.json').exists():
        # Never silently start over if the previous run has partial files without a committed latest.pt.
        raise RuntimeError(f'Found existing history/best without latest.pt in {folder}; manual recovery required')

    completed_file=folder/'train_complete.json'
    if completed_file.exists():
        completion=json.loads(completed_file.read_text())
        assert start == int(completion['epochs'])+1, f'Completion and latest.pt mismatch: {key}'
        assert (folder/'best_model.pt').exists(),f'Missing best_model.pt: {key}'
        print('SKIP completed training:',key,'after',completion['epochs'],'epochs')
        return

    # IMPORTANT: If switching notebooks just after an old checkpoint, evaluate existing history
    # before training any more epochs, without modifying or restarting past epochs.
    if history:
        best_file=folder/'best.json'
        assert best_file.exists() and (folder/'best_model.pt').exists(),f'Missing best state: {key}'
        best=json.loads(best_file.read_text())
        best_epoch=int(best['epoch'])
        assert any(int(row['epoch'])==best_epoch for row in history)
        stop_now,stale=early_stop_status(history,best_epoch)
        if stop_now:
            print('STOP FROM EXISTING HISTORY:',key,'last',start-1,
                  'best',best_epoch,'stale',stale,flush=True)
            mark_training_complete(folder,history,best,'early_stopping')
            return
        if start>EPOCHS:
            mark_training_complete(folder,history,best,'max_epochs')
            return

    for epoch in range(start,EPOCHS+1):
        start_time=time.time()
        # Exactly the original deterministic per-epoch setup; resume starts at epoch boundary.
        set_seed(seed+epoch)
        train_data=FacialDataset('train',augment=EXPERIMENTS[exp]['augmentation'],seed=seed,epoch=epoch)
        loader=make_loader(train_data,shuffle=True,seed=seed+epoch)
        model.train();total=0.;seen=0
        for batch_index,(x,y,p) in enumerate(loader):
            x=x.to(DEVICE,non_blocking=True)
            y=y.to(DEVICE,non_blocking=True)
            p=p.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits=model(x,p)
            loss=LOSS(logits,y)
            if not torch.isfinite(loss):raise RuntimeError(f'Nonfinite loss {key}, epoch {epoch}')
            loss.backward();optimizer.step()
            total+=float(loss.item())*len(x);seen+=len(x)
            if (batch_index+1)%500==0: print(key,'epoch',epoch,'trained',seen,'/',len(train_data),flush=True)
            if SMOKE_ONLY and batch_index>=1:break
        model.eval()
        val_data=FacialDataset('valid')
        if SMOKE_ONLY:
            val_data.names=val_data.names[:128]
        y_val,p_val=predict(model,val_data,desc=key+' validation')
        metrics=compute_metrics(y_val,p_val)
        row={'epoch':epoch,'training_loss':total/max(1,seen),'clean_validation_mAP':metrics['mAP'],
             'clean_validation_F1_at_05':metrics['macro_f1'],'seconds':time.time()-start_time}
        history.append(row)
        previous=folder/'best.json'
        if not previous.exists() or metrics['mAP']>json.loads(previous.read_text())['mAP']:
            write_torch(folder/'best_model.pt',{'model':model.state_dict(),'exp':exp,'seed':seed,'epoch':epoch})
            write_json(folder/'best.json',{'epoch':epoch,'mAP':metrics['mAP'],'F1_at_05':metrics['macro_f1']})
        # Keep original commit order. The latest.pt checkpoint marks each fully completed epoch.
        write_json(folder/'history.json',history)
        write_torch(latest,{'exp':exp,'seed':seed,'epoch':epoch,'model':model.state_dict(),
                            'optimizer':optimizer.state_dict(),'history':history})
        best=json.loads((folder/'best.json').read_text())
        stop_now,stale=early_stop_status(history,int(best['epoch']))
        print(key,'epoch',epoch,'train BCE',round(row['training_loss'],5),
              'clean-val mAP',round(metrics['mAP'],6),'best',best['epoch'],
              'stale epochs',stale,'early stop?',stop_now,flush=True)
        if SMOKE_ONLY:
            print('Smoke-only training exit. Do NOT regard this as a completed scientific run.')
            return
        if stop_now:
            mark_training_complete(folder,history,best,'early_stopping')
            return

    assert history and int(history[-1]['epoch'])==EPOCHS
    mark_training_complete(folder,history,json.loads((folder/'best.json').read_text()),'max_epochs')

if SMOKE_ONLY:
    # Never write short training into final study. Only perform model/data-loader sanity checks here.
    mini=FacialDataset('train',augment=True,seed=0,epoch=1)
    x,y,p=mini[0]
    assert x.shape==(3,224,224) and y.shape==(24,) and p.shape==(10,)
    assert torch.isfinite(x).all() and torch.isfinite(p).all()
    print('DATA SMOKE PASS: image',tuple(x.shape),'labels',tuple(y.shape),'geometry',tuple(p.shape))
else:
    for e in EXPERIMENTS:
        for seed in SEEDS:
            train_one(e,seed)
    print('All 18 training runs have completed.')

RESUME: A_seed0 completed epochs 6 next 7
SKIP completed training: A_seed0 after 6 epochs
RESUME: A_seed1 completed epochs 4 next 5
SKIP completed training: A_seed1 after 4 epochs
RESUME: A_seed2 completed epochs 4 next 5
SKIP completed training: A_seed2 after 4 epochs
RESUME: B_seed0 completed epochs 5 next 6
SKIP completed training: B_seed0 after 5 epochs
RESUME: B_seed1 completed epochs 4 next 5
SKIP completed training: B_seed1 after 4 epochs
RESUME: B_seed2 completed epochs 4 next 5
SKIP completed training: B_seed2 after 4 epochs
RESUME: C_seed0 completed epochs 6 next 7
SKIP completed training: C_seed0 after 6 epochs
RESUME: C_seed1 completed epochs 5 next 6
SKIP completed training: C_seed1 after 5 epochs
RESUME: C_seed2 completed epochs 6 next 7
SKIP completed training: C_seed2 after 6 epochs
RESUME: D_seed0 completed epochs 5 next 6
SKIP completed training: D_seed0 after 5 epochs
RESUME: D_seed1 completed epochs 3 next 4
D_seed1 epoch 4 trained 32000 / 162770
D_seed1 epoch 4 tra

In [ ]:
# 9 — Clean validation: export frozen thresholds for each independently trained run
# Runs after all trainings; no corrupted validation thresholds, no test access.
def load_best(exp,seed):
    folder=STUDY/f'{exp}_seed{seed}'
    assert (folder/'train_complete.json').exists(), f'Incomplete training: {folder}'
    obj=torch.load(folder/'best_model.pt',map_location='cpu',weights_only=True)
    model=make_model(exp,pretrained=False).to(DEVICE)
    model.load_state_dict(obj['model'],strict=True)
    model.eval()
    return model,folder

def export_thresholds(exp,seed):
    model,folder=load_best(exp,seed)
    outfile=folder/'clean_val_thresholds.json'
    if outfile.exists():
        print('SKIP existing frozen thresholds:',exp,seed)
        return
    truth,prob=predict(model,FacialDataset('valid'),desc=f'{exp} seed{seed} valid')
    thresholds=fit_thresholds(truth,prob)
    fixed=compute_metrics(truth,prob,thresholds)
    baseline=compute_metrics(truth,prob)
    write_json(outfile,{'thresholds':thresholds.tolist(), 'selected_epoch':json.loads((folder/'best.json').read_text())['epoch'],
                        'clean_validation_mAP':baseline['mAP'],
                        'clean_validation_F1_at_05':baseline['macro_f1'],
                        'clean_validation_F1_fitted_in_sample':fixed['macro_f1'],
                        'attribute_order':ATTRIBUTES})
    print('FROZEN THRESHOLDS:',exp,seed)
    del model
    if torch.cuda.is_available():torch.cuda.empty_cache()

if not SMOKE_ONLY:
    for e in EXPERIMENTS:
        for seed in SEEDS:
            export_thresholds(e,seed)
    print('All 18 frozen threshold files exist.')

A seed0 valid: 10368/19867
FROZEN THRESHOLDS: A 0
A seed1 valid: 10368/19867
FROZEN THRESHOLDS: A 1
A seed2 valid: 10368/19867
FROZEN THRESHOLDS: A 2
B seed0 valid: 10368/19867
FROZEN THRESHOLDS: B 0
B seed1 valid: 10368/19867
FROZEN THRESHOLDS: B 1
B seed2 valid: 10368/19867
FROZEN THRESHOLDS: B 2
C seed0 valid: 10368/19867
FROZEN THRESHOLDS: C 0
C seed1 valid: 10368/19867
FROZEN THRESHOLDS: C 1
C seed2 valid: 10368/19867
FROZEN THRESHOLDS: C 2
D seed0 valid: 10368/19867
FROZEN THRESHOLDS: D 0
D seed1 valid: 10368/19867
FROZEN THRESHOLDS: D 1
D seed2 valid: 10368/19867
FROZEN THRESHOLDS: D 2
E seed0 valid: 10368/19867
FROZEN THRESHOLDS: E 0
E seed1 valid: 10368/19867
FROZEN THRESHOLDS: E 1
E seed2 valid: 10368/19867
FROZEN THRESHOLDS: E 2
F seed0 valid: 10368/19867
FROZEN THRESHOLDS: F 0
F seed1 valid: 10368/19867
FROZEN THRESHOLDS: F 1
F seed2 valid: 10368/19867
FROZEN THRESHOLDS: F 2
All 18 frozen threshold files exist.


In [ ]:
# 10 — Final clean + 20-corruption test, only after full training & threshold export
# WARNING: the test data is held out; do not tune hyperparameters after inspecting results.

def condition_names():
    yield 'clean',None,0
    for family in FAMILIES:
        for severity in (1,2,3,4):
            yield f'{family}_s{severity}',family,severity

def test_one(exp,seed):
    model,folder=load_best(exp,seed)
    threshold_file=folder/'clean_val_thresholds.json'
    assert threshold_file.exists(),'Missing frozen thresholds; stop'
    threshold_record=json.loads(threshold_file.read_text())
    assert threshold_record['attribute_order']==ATTRIBUTES
    threshold=np.array(threshold_record['thresholds'])
    results_path=folder/'test_metrics.json'
    if results_path.exists():
        results=json.loads(results_path.read_text())
    else:
        results={}
    for label,family,severity in condition_names():
        if label in results:
            print('SKIP test:',exp,seed,label)
            continue
        ds=FacialDataset('test',family=family,severity=severity)
        assert len(ds)==19962
        truth,prob=predict(model,ds,desc=f'{exp} seed{seed} {label}')
        metrics=compute_metrics(truth,prob,threshold)
        metrics['F1_at_05']=compute_metrics(truth,prob)['macro_f1']
        results[label]=metrics
        write_json(results_path,results)    # safely resumes completed conditions
        print('TEST COMPLETE:',exp,seed,label,'mAP',round(metrics['mAP'],5),'F1',round(metrics['macro_f1'],5),flush=True)
    assert len(results)==21
    write_json(folder/'test_complete.json',{'conditions':21,'run':f'{exp}_seed{seed}'})
    del model
    if torch.cuda.is_available():torch.cuda.empty_cache()

if SMOKE_ONLY:
    print('SKIP official test in SMOKE_ONLY mode')
elif RUN_FINAL_TEST:
    assert all((STUDY/f'{e}_seed{s}'/'clean_val_thresholds.json').exists() for e in EXPERIMENTS for s in SEEDS)
    assert all((IMAGE_ROOT/n).is_file() for n in SPLITS['test']), 'Test images are not extracted'
    for e in EXPERIMENTS:
        for s in SEEDS:
            test_one(e,s)
    print('All 18 official test evaluations complete.')
else:
    print('Training / thresholds ready; RUN_FINAL_TEST=False, official test not started.')

A seed0 clean: 10368/19962
TEST COMPLETE: A 0 clean mAP 0.81992 F1 0.75664
A seed0 blur_s1: 10368/19962
TEST COMPLETE: A 0 blur_s1 mAP 0.80663 F1 0.70825
A seed0 blur_s2: 10368/19962
TEST COMPLETE: A 0 blur_s2 mAP 0.75549 F1 0.54589
A seed0 blur_s3: 10368/19962
TEST COMPLETE: A 0 blur_s3 mAP 0.67319 F1 0.31371
A seed0 blur_s4: 10368/19962
TEST COMPLETE: A 0 blur_s4 mAP 0.56546 F1 0.14241
A seed0 brightness_s1: 10368/19962
TEST COMPLETE: A 0 brightness_s1 mAP 0.81895 F1 0.7536
A seed0 brightness_s2: 10368/19962
TEST COMPLETE: A 0 brightness_s2 mAP 0.81449 F1 0.74427
A seed0 brightness_s3: 10368/19962
TEST COMPLETE: A 0 brightness_s3 mAP 0.80233 F1 0.71652
A seed0 brightness_s4: 10368/19962
TEST COMPLETE: A 0 brightness_s4 mAP 0.76058 F1 0.57126
A seed0 rotation_s1: 10368/19962
TEST COMPLETE: A 0 rotation_s1 mAP 0.81315 F1 0.7394
A seed0 rotation_s2: 10368/19962
TEST COMPLETE: A 0 rotation_s2 mAP 0.80357 F1 0.71971
A seed0 rotation_s3: 10368/19962
TEST COMPLETE: A 0 rotation_s3 mAP 0.791

In [ ]:
# 11 — Reports: per-condition + mean±sample SD + paired effects (mAP and frozen F1)
from itertools import product

def collect_results():
    rows=[];by_run={}
    for exp,seed in product(EXPERIMENTS,SEEDS):
        path=STUDY/f'{exp}_seed{seed}'/'test_metrics.json'
        if not path.exists():continue
        data=json.loads(path.read_text())
        by_run[(exp,seed)]=data
        for condition,metrics in data.items():
            family='clean' if condition=='clean' else condition.rsplit('_s',1)[0]
            rows.append({'model':exp,'seed':seed,'condition':condition,'family':family,
                         'mAP':metrics['mAP'],'frozen_F1':metrics['macro_f1'],'F1_at_05':metrics['F1_at_05']})
    return pd.DataFrame(rows),by_run

if not SMOKE_ONLY:
    table,by_run=collect_results()
    if len(table):
        table.to_csv(STUDY/'condition_results.csv',index=False)
        agg=table[table.condition!='clean'].groupby(['model','seed'],as_index=False)[['mAP','frozen_F1']].mean()
        agg.to_csv(STUDY/'corruption_averages_by_seed.csv',index=False)
        summary=agg.groupby('model')[['mAP','frozen_F1']].agg(['mean','std'])
        summary.to_csv(STUDY/'corruption_summary.csv')
        print('Corruption average (all 20 conditions) by condition:')
        display(summary)
        pairs=[('B','A','Geometry under clean training'),('D','C','Geometry under corruption-aware training'),
               ('F','E','Geometry under RGB-branch masking')]
        contrast=[]
        for a,b,desc in pairs:
            for s in SEEDS:
                xa=agg[(agg.model==a)&(agg.seed==s)]
                xb=agg[(agg.model==b)&(agg.seed==s)]
                if len(xa)==len(xb)==1:
                    contrast.append({'comparison':f'{a}-{b}','description':desc,'seed':s,
                                     'delta_mAP_pp':100*(xa.iloc[0]['mAP']-xb.iloc[0]['mAP']),
                                     'delta_F1_pp':100*(xa.iloc[0]['frozen_F1']-xb.iloc[0]['frozen_F1'])})
        contrast=pd.DataFrame(contrast)
        if len(contrast):
            contrast.to_csv(STUDY/'paired_seed_contrasts.csv',index=False)
            print('Paired geometry effects (percentage points):')
            display(contrast)
            if len(contrast)==9:
                idx=contrast.set_index(['comparison','seed'])
                interactions=[]
                for s in SEEDS:
                    for name,new,old in [('Training distribution','D-C','B-A'),('RGB masking','F-E','B-A')]:
                        interactions.append({'interaction':name,'seed':s,
                         'delta_mAP_pp':idx.loc[(new,s),'delta_mAP_pp']-idx.loc[(old,s),'delta_mAP_pp'],
                         'delta_F1_pp':idx.loc[(new,s),'delta_F1_pp']-idx.loc[(old,s),'delta_F1_pp']})
                interactions=pd.DataFrame(interactions)
                interactions.to_csv(STUDY/'paired_interactions.csv',index=False)
                print('Interaction-of-geometry-benefits:')
                display(interactions)
        print('All reports written to:',STUDY)
    else:print('No official test metrics yet; nothing to report.')

Corruption average (all 20 conditions) by condition:


mAP           frozen_F1          
           mean       std      mean       std
model                                        
A      0.760485  0.002264  0.613690  0.000473
B      0.759750  0.001425  0.619108  0.001386
C      0.797764  0.001206  0.734217  0.001624
D      0.798201  0.002106  0.735461  0.000848
E      0.757980  0.005573  0.612795  0.003728
F      0.759668  0.003360  0.611292  0.001407

Paired geometry effects (percentage points):


,comparison,description,seed,delta_mAP_pp,delta_F1_pp
0,B-A,Geometry under clean training,0,-0.395839,0.611207
1,B-A,Geometry under clean training,1,0.137236,0.611591
2,B-A,Geometry under clean training,2,0.038235,0.402458
3,D-C,Geometry under corruption-aware training,0,0.317474,0.109484
4,D-C,Geometry under corruption-aware training,1,-0.149020,0.324632
5,D-C,Geometry under corruption-aware training,2,-0.037416,-0.060945
6,F-E,Geometry under RGB-branch masking,0,-0.099280,-0.659576
7,F-E,Geometry under RGB-branch masking,1,0.275930,-0.050442
8,F-E,Geometry under RGB-branch masking,2,0.329628,0.259122


Interaction-of-geometry-benefits:


,interaction,seed,delta_mAP_pp,delta_F1_pp
0,Training distribution,0,0.713313,-0.501723
1,RGB masking,0,0.296558,-1.270783
2,Training distribution,1,-0.286256,-0.286959
3,RGB masking,1,0.138693,-0.662033
4,Training distribution,2,-0.075651,-0.463403
5,RGB masking,2,0.291393,-0.143337


All reports written to: /content/drive/MyDrive/FacialVisualProfile/runs/rq4_training_v1


## Important scientific and operational limitations

- **Status:** code generated from actual archived Drive source, but end-to-end 18-run training and final Colab GPU test must still be executed by you. This notebook does not claim to have reproduced existing benchmarks.
- **Data:** requires the original `FacialVisualProfile/data/` files in the same Google Drive account, and enough Colab local storage for the ZIP plus extracted images. Drive access must be authorized at runtime.
- **Difference from historical protocol:** Augmentation and whole-RGB branch masking are *new interventions*. Original clean-only pipeline used no augmentation. Current new runs are internally comparable but are not byte-for-byte reproductions of the historical seeded notebooks.
- **Corruption reproduction:** the recorded severities, JPEG settings, signed rotation rule and pixel-center landmarks are represented. The full-image pixel hashes should be checked against the original frozen benchmark implementation before claiming exact reproduction across Pillow/libjpeg versions. The best practice is to compare per-condition image hashes on a small deterministic sample against old frozen input records.
- **Selected models:** the CNN is fully fine-tuned; each run is initialized fresh. ImageNet pretrained weights may download into the Colab runtime if not cached.
- **Feature masking:** 20% of samples have all 512 RGB features set to zero during E/F *training*; no scaling is applied and masking is disabled in validation/test. It does not manually change weights or force good test-time reliance on geometry.
- **Resume:** checkpoint boundaries are *epochs*, not individual batches. A disconnected partial epoch will repeat. A stable Colab software/GPU environment is recommended for resumed epochs. `latest.pt` includes optimizer state; `best_model.pt` contains only the best epoch model weights. Do not run two Notebook sessions that write to the same study folder simultaneously.
- **No test tuning:** keep official Test reserved for final reporting. No p-values, no confidence claims from just three Seeds. Always report both the mean and all paired seed differences.
- **H2:** if real GT Geometry provides no extra advantage relative to R18 across these controls, that does not prove that the five points contain no information. Additional matched-architecture negative controls (Constant Geometry or shuffled landmarks) remain optional and are not part of this agreed 18-run plan.

### Resume from earlier 10-epoch version (important)

1. **Do not delete, rename or clear** `FacialVisualProfile/runs/rq4_training_v1/`. This version preserves `RUN_STUDY` and `EPOCHS=10` exactly; `EPOCHS` is now a maximum, not a mandatory count.
2. In the old running Colab, **wait for the full epoch summary** (e.g. `A_seed0 epoch 3 train BCE ... best ...`), then interrupt the *old* run. Do not simultaneously execute both notebooks. A mid-epoch interruption loses only the unfinished epoch; last completed `latest.pt` stays.
3. Open this notebook, mount the **same Google Drive account**, leave `SMOKE_ONLY=False`, run from the first cell. It prints `RESUME: A_seed0 completed epochs N next N+1` and retains the original `best_model.pt`, `best.json`, `history.json` and `latest.pt`.
4. `train_complete.json` can now record fewer than ten epochs. Downstream validation/threshold/test cells continue to work without other modifications. Already-completed 10-epoch runs remain valid and skipped.
5. This policy was introduced after some experiments began. `early_stopping_policy.json` documents it, and all not-yet-completed runs follow the same stop rule. A clean-validation criterion may not optimize corruption robustness. Do not change patience/minimum based on test results.
